In [1]:
import pandas as pd 
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string
from sklearn.preprocessing import FunctionTransformer
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_curve
)
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler 
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
data=pd.read_csv("F:\Git\deploy-proj\email-spam-clf\model\data\spam_data.csv")
data.head(3)

,label,text,num_characters,num_words,num_sentences,transformed_text
0,1,ounce feather bowl hummingbird opec moment ala...,148,20,1,ounc feather bowl hummingbird opec moment alab...
1,1,wulvob get your medircations online qnb ikud v...,808,104,1,wulvob get medirc onlin qnb ikud viagra escape...
2,0,computer connection from cnn com wednesday es...,2235,338,1,comput connect cnn com wednesday escapenumb ma...


In [3]:
spam_data=data.copy()

In [4]:
spam_data.shape

(83446, 6)

In [5]:
spam_data.isnull().sum()

label                0
text                 0
num_characters       0
num_words            0
num_sentences        0
transformed_text    19
dtype: int64

In [6]:
spam_data.dropna(subset=['transformed_text'], inplace=True)
print(spam_data.isnull().sum())

label               0
text                0
num_characters      0
num_words           0
num_sentences       0
transformed_text    0
dtype: int64


In [7]:
spam_data.duplicated(subset=['text']).sum()

np.int64(0)

In [8]:
duplicate_rows=spam_data[spam_data.duplicated(subset=['text'], keep=False)]
print(duplicate_rows.sort_values(by='text').head(10))

Empty DataFrame
Columns: [label, text, num_characters, num_words, num_sentences, transformed_text]
Index: []


In [ ]:
# Remove duplicates and keep only one of them
spam_data=spam_data.drop_duplicates(subset=['text'], keep='first')

In [ ]:
spam_data.duplicated(subset=['text']).sum()

In [9]:
spam_data.shape

(83427, 6)

In [9]:
spam_data['label'].value_counts()

label
1    43892
0    39535
Name: count, dtype: int64

In [ ]:
spam_data.info()

In [10]:
spam_data.isnull().sum()

label               0
text                0
num_characters      0
num_words           0
num_sentences       0
transformed_text    0
dtype: int64

In [11]:
x_train, x_test, y_train, y_test=train_test_split(spam_data['text'], spam_data['label'], test_size=0.2, random_state=42)

In [12]:
x_train

14422    attached are the historical numbers for the se...
28745    find out more about the new features and enhan...
82789    catatonic convocation hanoi cheekbone laredo f...
11863    invest in your future !\njoin us at one of pgs...
54182    mike ,\nmy number for next week is 66 ( from s...
                               ...                        
6265     anita . from our conversation today with daren...
54897    who visits knows what makes keynotes over stre...
76837    spring the day following hook show that wove o...
860      alternative medicine database over escapenumbe...
15797    ehud ,\ni shall send a message to jeff ' s sec...
Name: text, Length: 66741, dtype: object

In [13]:
y_train

14422    0
28745    1
82789    1
11863    0
54182    0
        ..
6265     0
54897    1
76837    1
860      0
15797    0
Name: label, Length: 66741, dtype: int64

In [27]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

# def transforms_text(text):
#     # Lowercase and tokenize
#     tokens = nltk.word_tokenize(text.lower())
    
#     transformed = []
#     for word in tokens:
#         # Remove standalone punctuation and stopwords
#         if word not in stop_words and word not in punctuation:
#             # Check if it has at least one alphanumeric character 
#             # (Allows "w1nn3r" but removes "!!!")
#             if any(char.isalnum() for char in word):
#                 transformed.append(ps.stem(word))
    
#     return " ".join(transformed)

def transforms_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase and tokenize
    tokens = nltk.word_tokenize(text.lower())
    
    transformed = []
    for word in tokens:
        # Keep words, keep numbers, and keep currency symbols
        # We only remove punctuation that is NOT a currency sign or number
        if word not in stop_words and (word.isalnum() or word in ['$']):
            # Optional: Remove stemming. 
            # Sometimes stemming "winning" to "win" loses the "urgency" of spam.
            # Let's keep stemming for now but ensure it doesn't kill the word.
            transformed.append(ps.stem(word))
    
    return " ".join(transformed)

In [28]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

# # 1. Wrap your existing function to handle arrays
# def text_cleaner_wrapper(text_list):
#     # Vectorize ensures your function runs on every string in the input list
#     clean_func = np.vectorize(transforms_text)
#     return clean_func(text_list)

def text_cleaner_wrapper(text_input):
    """
    Processes a list or Series of strings one by one to avoid 
    the massive memory allocation spike of np.vectorize.
    """
    # If the input is a single string (from a single prediction), 
    # wrap it in a list so the loop works.
    if isinstance(text_input, str):
        text_input = [text_input]
        
    # Standard Python list comprehension: low memory overhead
    return [transforms_text(text) for text in text_input]

# 2. Update the Pipeline
final_pipeline = Pipeline([
    ('cleaner', FunctionTransformer(text_cleaner_wrapper)), # Your NLTK logic first
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2), 
        use_idf=True,
        max_features=10000, # Increased
        token_pattern=r"(?u)\b\w\w+\b|\b\d+\b|[$]" # Sees words, numbers, and $
    )),
    ('scaler', StandardScaler(with_mean=False)),
    ('clf', SGDClassifier(loss='log_loss', alpha=0.01, random_state=42))
])

# 3. Train on RAW text (X_train['text'] instead of 'transformed_text')
final_pipeline.fit(x_train, y_train)

,steps,"[('cleaner', ...), ('tfidf', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function tex...00231930100E0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [29]:
y_pred=final_pipeline.predict(x_test)

In [30]:
# from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[7771  113]
 [ 130 8672]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      7884
           1       0.99      0.99      0.99      8802

    accuracy                           0.99     16686
   macro avg       0.99      0.99      0.99     16686
weighted avg       0.99      0.99      0.99     16686



In [31]:
# A few test cases
test_emails = [
    "Hey, are we still meeting for the AIML group study at 5?", # Should be Ham
    "CONGRATULATIONS! You've won a $1000 Walmart gift card. Click here to claim now!", # Should be Spam
    "Please find the attached invoice for your recent cloud subscription." # Should be Ham
]

# The pipeline handles the raw text automatically!
predictions = final_pipeline.predict(test_emails)
probabilities = final_pipeline.predict_proba(test_emails)

for email, pred, prob in zip(test_emails, predictions, probabilities):
    label = "SPAM" if pred == 1 else "HAM"
    conf = max(prob) * 100
    print(f"Result: {label} ({conf:.2f}% confidence) | Text: {email[:50]}...")

Result: SPAM (56.96% confidence) | Text: Hey, are we still meeting for the AIML group study...
Result: SPAM (98.00% confidence) | Text: CONGRATULATIONS! You've won a $1000 Walmart gift c...
Result: HAM (99.90% confidence) | Text: Please find the attached invoice for your recent c...


In [ ]:
# import numpy as np
# from sklearn.preprocessing import FunctionTransformer

# # 1. Wrap your existing function to handle arrays
# # def text_cleaner_wrapper(text_list):
# #     # Vectorize ensures your function runs on every string in the input list
# #     clean_func = np.vectorize(transforms_text)
# #     return clean_func(text_list)

# def text_cleaner_wrapper(text_input):
#     """
#     Processes a list or Series of strings one by one to avoid 
#     the massive memory allocation spike of np.vectorize.
#     """
#     # If the input is a single string (from a single prediction), 
#     # wrap it in a list so the loop works.
#     if isinstance(text_input, str):
#         text_input = [text_input]
        
#     # Standard Python list comprehension: low memory overhead
#     return [transforms_text(text) for text in text_input]

# # 2. Update the Pipeline
# final_pipeline = Pipeline([
#     ('cleaner', FunctionTransformer(text_cleaner_wrapper)), # Your NLTK logic first
#     ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)), 
#     ('scaler', StandardScaler(with_mean=False)),
#     ('clf', SGDClassifier(loss='log_loss', alpha=0.01, random_state=42))
# ])

# # 3. Train on RAW text (X_train['text'] instead of 'transformed_text')
# final_pipeline.fit(x_train, y_train)

In [42]:
sample_mail = "free http money"

# Step 1: See what the cleaner does
cleaned_version = text_cleaner_wrapper(sample_mail)
print(f"Cleaned version: {cleaned_version}")

# Step 2: See what the model predicts for that specific cleaned version
direct_pred = final_pipeline.predict([sample_mail])
print(f"Pipeline Prediction: {direct_pred}")

Cleaned version: ['free http money']
Pipeline Prediction: [1]


In [33]:
# Check what the model thinks are the top "Spam" words
feature_names = final_pipeline.named_steps['tfidf'].get_feature_names_out()
coefs = final_pipeline.named_steps['clf'].coef_[0]
top_spam = sorted(zip(coefs, feature_names), reverse=True)[:15]
print("Top Spam Words:", top_spam)

Top Spam Words: [(np.float64(0.18292184033276868), 'http'), (np.float64(0.1400249264880785), 'escapelong'), (np.float64(0.126099416239154), 'escapelong escapelong'), (np.float64(0.1198977301863411), '2004'), (np.float64(0.11305276049854253), 'attach http'), (np.float64(0.11254867065455168), 'hk'), (np.float64(0.11147921154186219), 'info'), (np.float64(0.10837888491618342), 'com'), (np.float64(0.10421244059368205), 'messag'), (np.float64(0.10289483601718069), '2005'), (np.float64(0.09812767073218955), 'remov'), (np.float64(0.09742553955518664), 'medic'), (np.float64(0.09490523277765646), 'onlin'), (np.float64(0.09201356079461691), 'recipi'), (np.float64(0.08467188076925954), 'buy')]


In [34]:
print(final_pipeline.predict(["Check out this http link for online medic"]))

[1]


In [1]:
import pandas as pd

In [2]:
data=pd.read_csv(r"D:\Complete_proj\Email_spam\model\data\gemini_data.csv")

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 948 entries, 0 to 947
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  948 non-null    object
 1   Message   948 non-null    object
dtypes: object(2)
memory usage: 14.9+ KB
